![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [2]:
# QuantBook Analysis Tool
# For more information see [https://www.quantconnect.com/docs/v2/our-platform/research/getting-started]
qb = QuantBook()
spy = qb.add_equity("SPY")
history = qb.history(qb.securities.keys(), 360, Resolution.DAILY)

# Indicator Analysis
bbdf = qb.indicator(BollingerBands(30, 2), spy.symbol, 360, Resolution.DAILY)
bbdf.drop('standarddeviation', axis=1).plot()

In [5]:
from keras.layers import LSTM, Dense, Dropout
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from datetime import datetime


# Instantiate a QuantBook.
qb = QuantBook()

# Select the desired index for research.
asset = "SPY"
# Call the add_equity method with the tickers, and their corresponding resolution.
qb.AddEquity(asset, Resolution.MINUTE)
# If you do not pass a resolution argument, Resolution.MINUTE is used by default.
# Call the history method with qb.securities.keys for all tickers, time argument(s), and resolution to request historical data for the symbol.
history = qb.History(qb.Securities.Keys, datetime(2015, 1, 1), datetime(2025, 6, 30), Resolution.DAILY)

# Select the close column and then call the unstack method.
close_price = history['close'].unstack(level=0)

# Initialize MinMaxScaler to scale the data onto [0,1].
scaler = MinMaxScaler(feature_range=(0, 1))
# Transform our data 
df = pd.DataFrame(scaler.fit_transform(close_price), index=close_price.index)

# Select input data 
input_ = df.copy()

# Initialize another scaler for the output if needed, or reuse
output_scaler = MinMaxScaler(feature_range=(0, 1))
# Shift the data for 1-step backward as training output result.
output = df.shift(-1).iloc[:-1]
input_ = input_.iloc[:-1]  # Also trim input to match output length

# Split the data into training and testing sets.
# In this example, we use the first 80% data for training, and the last 20% for testing.
splitter = int(input_.shape[0] * 0.8)
X_train = input_.iloc[:splitter]
X_test = input_.iloc[splitter:]
y_train = output.iloc[:splitter]
y_test = output.iloc[splitter:]

# Build feature and label sets (using number of steps 60, and feature rank 1).
features_set = []
labels = []
for i in range(60, X_train.shape[0]):
    features_set.append(X_train.iloc[i-60:i].values.reshape(-1, 1))
    labels.append(y_train.iloc[i])
features_set, labels = np.array(features_set), np.array(labels)
features_set = np.reshape(features_set, (features_set.shape[0], features_set.shape[1], 1))

# Build a Sequential keras model.
model = Sequential()
# Create the model infrastructure.
# Add our first LSTM layer - 50 nodes.
model.add(LSTM(units=50, return_sequences=True, input_shape=(features_set.shape[1], 1)))
# Add Dropout layer to avoid overfitting
model.add(Dropout(0.2))
# Add additional layers
model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(units=50))
model.add(Dropout(0.2))
model.add(Dense(units=5))
model.add(Dense(units=1))
# Compile the model. We use Adam as optimizer for adaptive step size and MSE as loss function since it is continuous data.
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae', 'acc'])
# Set early stopping callback method.
callback = EarlyStopping(monitor='loss', patience=3, verbose=1, restore_best_weights=True)
# Display the model structure.
model.summary()
# Fit the model to our data, running 20 training epochs.
# Note that different training session's results will not be the same since the batch is randomly selected.
model.fit(features_set, labels, epochs=20, batch_size=100, callbacks=[callback])

# Get testing set features for input.
test_features = []
for i in range(60, X_test.shape[0]):
    test_features.append(X_test.iloc[i-60:i].values.reshape(-1, 1))
test_features = np.array(test_features)
test_features = np.reshape(test_features, (test_features.shape[0], test_features.shape[1], 1))
# Make predictions.
predictions = model.predict(test_features)
# Transform predictions back to original data-scale.
predictions = scaler.inverse_transform(predictions)
actual = scaler.inverse_transform(y_test.values[60:])  

# Plot the results.
plt.figure(figsize=(15, 10))
plt.plot(actual, color='blue', label='Actual')
plt.plot(predictions, color='red', label='Prediction')
plt.title('Price vs Predicted Price')
plt.legend()
plt.show()

In [ ]:
from keras.layers import LSTM, Dense, Dropout
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from datetime import datetime

qb = QuantBook()

# Get data
spy = qb.AddEquity("SPY", Resolution.DAILY).Symbol
history = qb.History([spy], datetime(2015, 1, 1), datetime(2025, 6, 30), Resolution.DAILY)
df = history['close'].unstack(level=0)['SPY'].to_frame(name='close')

# Split training and testing data
train_data = df.loc['2015-01-01':'2021-12-31']
test_data  = df.loc['2022-01-01':]

print(f"Train shape: {train_data.shape} | Test shape: {test_data.shape}")

# Scaling
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_data.values)                          

train_scaled = scaler.transform(train_data.values)
test_scaled  = scaler.transform(test_data.values)

# Convert to DataFrame with dates for easy slicing
train_df = pd.DataFrame(train_scaled, index=train_data.index, columns=['close'])
test_df  = pd.DataFrame(test_scaled,  index=test_data.index,  columns=['close'])

# Create sequences (90-day lookback) 
def create_sequences(data, seq_length=90):
    X, y = [], []
    for i in range(seq_length, len(data) - 1):        
        X.append(data[i - seq_length : i, 0])         
        y.append(data[i + 1, 0])                          
    return np.array(X), np.array(y)

seq_length = 90

X_train, y_train = create_sequences(train_scaled, seq_length)
X_test,  y_test  = create_sequences(test_scaled,  seq_length)

# Reshape for LSTM [samples, timesteps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test  = X_test.reshape((X_test.shape[0],  X_test.shape[1],  1))

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# Build LSTM model
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(seq_length, 1)))
model.add(Dropout(0.2))
model.add(LSTM(50, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(50))
model.add(Dropout(0.2))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')
model.summary()

# Train 
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train,
                    epochs=100,
                    batch_size=32,
                    validation_split=0.1,
                    callbacks=[early_stop],
                    verbose=1)

# Predict on test set 
pred_scaled = model.predict(X_test)

# Inverse transform to real prices
pred = scaler.inverse_transform(pred_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Align dates (skip first 60 days of test period used for sequence)
test_dates = test_data.index[seq_length:seq_length + len(pred)]

# Evaluation 
mse = np.mean((pred - y_test_actual) ** 2)
print(f"\n=== FINAL RESULT (Only SPY Price LSTM) ===")
print(f"Test MSE (2022-2025): {mse:.2f}")
print(f"Test RMSE: ${np.sqrt(mse):.2f} per day")

# Plot 
plt.figure(figsize=(16, 8))
plt.plot(test_dates, y_test_actual, label='Actual SPY Price', color='blue')
plt.plot(test_dates, pred, label=f'Predicted (MSE = {mse:.1f})', color='red', alpha=0.8)
plt.title('SPY Next-Day Price Prediction - Fixed LSTM (Only SPY Data)')
plt.xlabel('Date')
plt.ylabel('SPY Close Price')
plt.legend()
plt.grid(alpha=0.3)
plt.show()